# LeetCode 787 — Cheapest Flights Within K Stops bằng Python

Link đề: https://leetcode.com/problems/cheapest-flights-within-k-stops/ · Độ khó: **Medium** · Chủ đề: Graph, Dynamic Programming, Shortest Path, Breadth-First Search, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def findCheapestPrice(self, n: int, flights: List[List[int]], src: int, dst: int, k: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có `n` thành phố và các chuyến bay **một chiều** có giá; tìm **giá rẻ nhất** từ `src` đến `dst` với **tối đa `k` điểm dừng** (tức tối đa **`k + 1` chuyến bay**), không có thì trả `-1`.
2. Nắm các công cụ Python: danh sách cạnh, danh sách kề bằng `defaultdict(list)`, `float("inf")`, sao chép list (`dist[:]`), `collections.deque`, hàng đợi ưu tiên `heapq`, `functools.lru_cache`.
3. Đi từ **vét cạn DFS mọi hành trình** → **Bellman-Ford giới hạn `k + 1` vòng** → **BFS theo tầng (mỗi tầng = một chuyến bay)** → **Dijkstra có thêm "số chuyến bay đã dùng" trong trạng thái** → **quy hoạch động đệ quy có nhớ (top-down)**.
4. Hiểu mấu chốt: ngoài **giá**, bài còn bị ràng buộc **số cạnh**; vì vậy "đường rẻ nhất đến một thành phố" **không đủ** — ta phải nhớ **giá rẻ nhất khi dùng tối đa `t` chuyến** cho từng `t`, và Bellman-Ford theo từng vòng làm đúng điều đó.
5. Tránh các lỗi kinh điển: dùng Dijkstra thường (bỏ qua số điểm dừng), quên sao chép mảng `dist` giữa các vòng Bellman-Ford (đi nhiều chuyến hơn cho phép), nhầm `k` điểm dừng với `k` chuyến bay (lệch 1), đánh dấu "đã thăm" làm mất đường ít chuyến hơn, quên đổi `inf` thành `-1`.

## 2. Đề bài & ví dụ

Có `n` thành phố được nối bởi một số chuyến bay. Cho mảng `flights` trong đó `flights[i] = [from_i, to_i, price_i]` nghĩa là có chuyến bay **từ** thành phố `from_i` **đến** thành phố `to_i` với giá `price_i`.

Cho ba số nguyên `src`, `dst`, `k`. Hãy trả về **giá rẻ nhất** để đi từ `src` đến `dst` với **tối đa `k` điểm dừng** (điểm dừng là các thành phố **trung gian**, không tính `src` và `dst`). Nếu không có hành trình nào, trả về `-1`.

Chú ý: **tối đa `k` điểm dừng** ⇔ **tối đa `k + 1` chuyến bay**.

**Ví dụ 1**

- Input: `n = 4`, `flights = [[0,1,100],[1,2,100],[2,0,100],[1,3,600],[2,3,200]]`, `src = 0`, `dst = 3`, `k = 1`
- Output: `700`
- Giải thích: hành trình tối ưu với tối đa 1 điểm dừng là `0 → 1 → 3` giá `100 + 600 = 700`. Đường `0 → 1 → 2 → 3` rẻ hơn (`400`) nhưng dùng 2 điểm dừng nên **không hợp lệ**.

**Ví dụ 2**

- Input: `n = 3`, `flights = [[0,1,100],[1,2,100],[0,2,500]]`, `src = 0`, `dst = 2`, `k = 1`
- Output: `200` (đường `0 → 1 → 2`).

**Ví dụ 3**

- Input: cùng đồ thị ví dụ 2 nhưng `k = 0`
- Output: `500` (chỉ được bay thẳng).

**Ràng buộc:**

- `2 <= n <= 100`; `0 <= flights.length <= n * (n - 1) / 2`
- `flights[i].length == 3`; `0 <= from_i, to_i < n`; `from_i != to_i`
- `1 <= price_i <= 10^4`; không có hai chuyến bay trùng giữa cùng hai thành phố
- `0 <= src, dst, k < n`; `src != dst`

In [1]:
from typing import List
from collections import defaultdict, deque
from functools import lru_cache
import heapq
import itertools
import random
import time

INF = float("inf")

examples = [
    (4, [[0, 1, 100], [1, 2, 100], [2, 0, 100], [1, 3, 600], [2, 3, 200]], 0, 3, 1, 700),
    (3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 1, 200),
    (3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 0, 500),
]
for n, flights, src, dst, k, expected in examples:
    print(f"n = {n}, src = {src}, dst = {dst}, k = {k} -> kỳ vọng {expected}")

n = 4, src = 0, dst = 3, k = 1 -> kỳ vọng 700
n = 3, src = 0, dst = 2, k = 1 -> kỳ vọng 200
n = 3, src = 0, dst = 2, k = 0 -> kỳ vọng 500


## 3. Kiến thức nền

### 3.1. Đồ thị có hướng, có trọng số

Mỗi thành phố là một **đỉnh** (đánh số `0..n-1`), mỗi chuyến bay là một **cạnh có hướng** `from → to` với **trọng số** `price`. Ta có thể lưu bằng **danh sách cạnh** (như đề cho) hoặc **danh sách kề**: với mỗi đỉnh `u`, danh sách các cặp `(v, price)` đi ra từ `u`.

In [2]:
flights = examples[0][1]
adj = defaultdict(list)
for u, v, w in flights:
    adj[u].append((v, w))
for u in range(4):
    print(f"từ {u}:", adj[u])
print("từ 3 (không có chuyến đi):", adj[3])

từ 0: [(1, 100)]
từ 1: [(2, 100), (3, 600)]
từ 2: [(0, 100), (3, 200)]
từ 3: []
từ 3 (không có chuyến đi): []


### 3.2. "Điểm dừng" và "chuyến bay"

Một hành trình `src → a → b → dst` có **2 điểm dừng** (`a`, `b`) và **3 chuyến bay**. Vậy `k` điểm dừng ⇔ tối đa `k + 1` cạnh. Đây là chỗ rất dễ lệch 1.

### 3.3. Vô cực `float("inf")` và phép `min`

Để biểu diễn "chưa đến được", ta dùng `float("inf")`: `inf + 5 = inf`, `min(inf, 7) = 7`, `inf > mọi số`. Cuối cùng nếu kết quả vẫn là `inf` thì đổi thành `-1`.

In [3]:
print(INF + 5, min(INF, 7), INF > 10**18, INF == INF)
dist = [INF] * 4
dist[0] = 0
print(dist, "-> các thành phố chưa tới được là inf")

inf 7 True True
[0, inf, inf, inf] -> các thành phố chưa tới được là inf


### 3.4. Sao chép list: `dist[:]` và vì sao quan trọng

`b = a` **không** sao chép mà chỉ cho `b` trỏ cùng một list; `b = a[:]` (hoặc `list(a)`) tạo bản sao độc lập. Ở Bellman-Ford theo vòng, ta cần "bản chụp" của mảng `dist` ở **đầu vòng** để đảm bảo mỗi vòng chỉ thêm **đúng một chuyến bay**.

In [4]:
a = [1, 2, 3]
b = a            # cùng một list
c = a[:]         # bản sao
a[0] = 99
print("a =", a, "| b =", b, "| c =", c)

a = [99, 2, 3] | b = [99, 2, 3] | c = [1, 2, 3]


### 3.5. `heapq` — hàng đợi ưu tiên (min-heap)

`heapq.heappush(h, item)` thêm, `heapq.heappop(h)` lấy **phần tử nhỏ nhất**. Với tuple, Python so sánh lần lượt từng thành phần, nên `(cost, node, edges)` được lấy theo `cost` nhỏ nhất trước. Dijkstra dựa vào đó.

In [5]:
h = []
for item in [(500, 2, 1), (100, 1, 1), (200, 3, 2), (100, 0, 0)]:
    heapq.heappush(h, item)
print([heapq.heappop(h) for _ in range(4)])

[(100, 0, 0), (100, 1, 1), (200, 3, 2), (500, 2, 1)]


### 3.6. `deque` và `lru_cache`

`deque` là hàng đợi hai đầu (`append`, `popleft` đều `O(1)`) dùng cho BFS. `functools.lru_cache` biến một hàm đệ quy thành **có nhớ** (memoization): kết quả cho cùng đối số chỉ tính một lần.

In [6]:
@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(50), fib.cache_info().hits, "lần dùng lại kết quả đã nhớ")
q = deque([1])
q.append(2)
print(q.popleft(), list(q))

12586269025 48 lần dùng lại kết quả đã nhớ
1 [2]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Xét ví dụ 1 với `k = 1` (tối đa **2 chuyến bay**).

```
        100        100
   0 ---------> 1 ---------> 2
   ^            | 600        | 200
   |            v            v
   +---------- (3) <---------+
        100 (2 -> 0)
```

Hãy hỏi: "với **đúng/tối đa `t` chuyến bay**, giá rẻ nhất đến từng thành phố là bao nhiêu?" (xuất phát `0`):

| số chuyến tối đa `t` | giá tới `0` | `1` | `2` | `3` |
|---|---|---|---|---|
| 0 | 0 | inf | inf | inf |
| 1 | 0 | 100 | inf | inf |
| 2 | 0 | 100 | 200 | **700** |
| 3 | 0 | 100 | 200 | **400** |

Cột cuối: với 2 chuyến (`k = 1`) đến `3` rẻ nhất là `700` (`0→1→3`); phải có **3 chuyến** (`k = 2`) mới đạt `400` (`0→1→2→3`).

**Nhận xét quan trọng:** giá đến một thành phố **thay đổi theo số chuyến được phép**. Vì vậy một lời giải "đường rẻ nhất" như Dijkstra thông thường (chỉ lưu một `dist[v]`) sẽ trả `400` — sai. Ta cần xử lý **từng mức số chuyến**.

In [7]:
n, flights, src, dst, k, _ = examples[0]
dist = [INF] * n
dist[src] = 0
print(f"t = 0: {dist}")
for t in range(1, 4):
    nxt = dist[:]
    for u, v, w in flights:
        if dist[u] + w < nxt[v]:
            nxt[v] = dist[u] + w
    dist = nxt
    print(f"t = {t}: {dist}")

t = 0: [0, inf, inf, inf]
t = 1: [0, 100, inf, inf]
t = 2: [0, 100, 200, 700]
t = 3: [0, 100, 200, 400]


## 5. Cách 1: Brute Force (DFS vét cạn mọi hành trình)

### Ý tưởng

Làm đúng theo đề: thử **mọi hành trình** bắt đầu ở `src` với tối đa `k + 1` chuyến bay; mỗi lần tới `dst` thì ghi nhận tổng giá; lấy giá nhỏ nhất. Ta cho phép đi lại thành phố đã qua (giá dương nên hành trình tối ưu không lặp, nhưng vét cạn không cần lo).

Cài đặt: `dfs(u, flights_left, cost)`:

- Nếu `u == dst`: cập nhật `best = min(best, cost)` (có thể dừng tại đây vì đi tiếp chỉ tốn thêm).
- Nếu `flights_left == 0`: dừng.
- Ngược lại: thử mọi chuyến `u → v` với `flights_left - 1`.

In [8]:
def cheapest_brute(n, flights, src, dst, k):
    adj = defaultdict(list)
    for u, v, w in flights:
        adj[u].append((v, w))
    best = INF

    def dfs(u, left, cost):
        nonlocal best
        if u == dst:
            best = min(best, cost)
            return
        if left == 0:
            return
        for v, w in adj[u]:
            dfs(v, left - 1, cost + w)

    dfs(src, k + 1, 0)
    return -1 if best == INF else best


for n, flights, src, dst, k, expected in examples:
    got = cheapest_brute(n, flights, src, dst, k)
    print(got, "(kỳ vọng", expected, ")")
    assert got == expected

700 (kỳ vọng 700 )
200 (kỳ vọng 200 )
500 (kỳ vọng 500 )


### Độ phức tạp

- Mỗi đỉnh có tới `n - 1` chuyến đi ra, độ sâu `k + 1` ⇒ cỡ `O((n - 1)^(k + 1))` hành trình: **mũ** theo `k`. Với `n = 100`, `k = 99` là không thể.
- Space `O(k)` cho ngăn xếp đệ quy.

### Nút thắt

Cùng một cặp `(thành phố, số chuyến còn lại)` bị khám phá **lặp đi lặp lại** theo nhiều đường khác nhau, và mỗi lần ta lại duyệt lại toàn bộ phần phía sau. Đo số lần gọi `dfs` trên đồ thị đầy đủ:

In [9]:
def count_dfs_calls(n, k):
    """Số lần gọi dfs trên đồ thị đầy đủ n đỉnh (mọi cặp có cạnh) với dst không tới được (đi hết độ sâu)."""
    adj = {u: [v for v in range(n) if v != u] for u in range(n)}
    calls = 0

    def dfs(u, left):
        nonlocal calls
        calls += 1
        if left == 0:
            return
        for v in adj[u]:
            if v != n - 1:             # coi n-1 là dst nhưng bỏ qua để đi hết độ sâu
                dfs(v, left - 1)

    dfs(0, k + 1)
    return calls

for n in (5, 6, 7):
    print(f"n = {n}:", [count_dfs_calls(n, k) for k in range(0, 6)])

n = 5: [4, 13, 40, 121, 364, 1093]
n = 6: [5, 21, 85, 341, 1365, 5461]
n = 7: [6, 31, 156, 781, 3906, 19531]


Số lần gọi tăng **nhân lên** mỗi khi `k` tăng thêm 1 (mũ). Cần cách ghi nhớ "tốt nhất đến mỗi thành phố sau `t` chuyến".

## 6. Ý tưởng thuật toán / bước đột phá

Bước đột phá là **chia theo số chuyến bay**. Gọi `dp[t][v]` là **giá rẻ nhất để tới `v` dùng tối đa `t` chuyến bay**. Khi đó:

- `dp[0][src] = 0`, các `dp[0][v]` khác `= inf`.
- `dp[t][v] = min( dp[t - 1][v], min over (u → v, giá w) của dp[t - 1][u] + w )`.

Đáp án là `dp[k + 1][dst]` (đổi `inf` thành `-1`). Mỗi mức `t` chỉ cần **duyệt các cạnh một lần** (`O(E)`), nên tổng `O(k * E)` — không còn mũ. Chỉ cần mức `t - 1` để tính mức `t`, nên có thể dùng hai mảng.

Từ ý tưởng "trạng thái = (thành phố, số chuyến đã dùng)" ta có nhiều cách:

- **Cách 2:** Bellman-Ford chạy đúng `k + 1` vòng (dp theo vòng).
- **Cách 3:** BFS theo tầng — tầng `t` là các thành phố vừa được cải thiện ở chuyến thứ `t`.
- **Cách 4:** Dijkstra trên đồ thị trạng thái `(thành phố, số chuyến)` với hàng đợi ưu tiên theo giá.
- **Cách 5:** đệ quy có nhớ `f(v, t)` (top-down) tính ngược từ `dst`.

In [10]:
# minh họa: dp[t][v] cho ví dụ 1 (t từ 0 đến 3)
n, flights, src, dst, k, _ = examples[0]
dp = [[INF] * n for _ in range(4)]
dp[0][src] = 0
for t in range(1, 4):
    dp[t] = dp[t - 1][:]                      # dùng ít chuyến hơn vẫn hợp lệ
    for u, v, w in flights:
        dp[t][v] = min(dp[t][v], dp[t - 1][u] + w)
for t, row in enumerate(dp):
    print(f"dp[{t}] =", row)
print("đáp án k = 1 -> dp[2][3] =", dp[k + 1][dst])

dp[0] = [0, inf, inf, inf]
dp[1] = [0, 100, inf, inf]
dp[2] = [0, 100, 200, 700]
dp[3] = [0, 100, 200, 400]
đáp án k = 1 -> dp[2][3] = 700


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Bellman-Ford giới hạn `k + 1` vòng (nộp)

**Giải thích từng đoạn:**

1. `dist = [inf] * n`, `dist[src] = 0`.
2. Lặp `k + 1` lần (mỗi lần cho phép thêm đúng **một** chuyến bay):
   - `prev = dist[:]` — **bản chụp** của mảng đầu vòng.
   - Với mỗi chuyến `(u, v, w)`: nếu `prev[u] + w < dist[v]` thì `dist[v] = prev[u] + w`. Ta **đọc từ `prev`**, **ghi vào `dist`**, để một vòng chỉ đi thêm một chuyến.
3. Kết quả `dist[dst]` (nếu `inf` thì `-1`).

**Code demo (in từng vòng):**

In [11]:
def bellman_ford_demo(n, flights, src, dst, k):
    dist = [INF] * n
    dist[src] = 0
    print("khởi tạo:", dist)
    for t in range(1, k + 2):
        prev = dist[:]
        for u, v, w in flights:
            if prev[u] + w < dist[v]:
                dist[v] = prev[u] + w
        print(f"sau vòng {t} (tối đa {t} chuyến):", dist)
    return -1 if dist[dst] == INF else dist[dst]

print("đáp án:", bellman_ford_demo(*examples[0][:5]))

khởi tạo: [0, inf, inf, inf]
sau vòng 1 (tối đa 1 chuyến): [0, 100, inf, inf]
sau vòng 2 (tối đa 2 chuyến): [0, 100, 200, 700]
đáp án: 700


**Code nộp LeetCode (cách 2):**

In [12]:
class Solution2:
    def findCheapestPrice(self, n: int, flights: List[List[int]], src: int, dst: int, k: int) -> int:
        INF_ = float("inf")
        dist = [INF_] * n
        dist[src] = 0
        for _ in range(k + 1):
            prev = dist[:]
            for u, v, w in flights:
                if prev[u] + w < dist[v]:
                    dist[v] = prev[u] + w
        return -1 if dist[dst] == INF_ else dist[dst]


for n, flights, src, dst, k, expected in examples:
    assert Solution2().findCheapestPrice(n, flights, src, dst, k) == expected
print("Cách 2 OK")

Cách 2 OK


**Độ phức tạp:** Time `O((k + 1) * E)` với `E` là số chuyến bay (tối đa `n(n-1)/2 ≈ 4950`, `k + 1 <= 100` ⇒ cỡ `5 * 10^5`). Space `O(n)`.

### 7.2. Cách 3 — BFS theo tầng (mỗi tầng một chuyến bay)

**Ý tưởng khác:** chỉ những thành phố **vừa được cải thiện giá** ở tầng trước mới có thể làm cho tầng sau tốt hơn. Vì vậy ta duyệt theo **tầng**: tầng `t` chứa các `(thành phố, giá)` vừa tìm được ở chuyến thứ `t`. Giữ `best[v]` là giá rẻ nhất đã biết; chỉ đẩy vào tầng sau khi tìm được giá **nhỏ hơn** `best[v]`.

**Giải thích từng đoạn:**

1. `best = [inf] * n`, `best[src] = 0`; hàng đợi `deque([(src, 0)])`.
2. Lặp tối đa `k + 1` tầng: lấy **toàn bộ** các phần tử của tầng hiện tại (chốt `size = len(queue)`), với mỗi `(u, cost)` duyệt chuyến `u → v, w`.
3. Nếu `cost + w < best[v]` thì cập nhật `best[v]` và đẩy `(v, cost + w)` cho tầng sau.
4. Điểm tinh tế: khi duyệt tầng, ta so với `best` **được cập nhật trong cùng tầng**; nhưng vì chỉ dùng `cost` của chính phần tử (đã chụp lúc đẩy vào) chứ không dùng `best[u]` hiện tại, nên mỗi tầng vẫn chỉ thêm đúng một chuyến.

**Code demo:**

In [13]:
def bfs_levels_demo(n, flights, src, dst, k):
    adj = defaultdict(list)
    for u, v, w in flights:
        adj[u].append((v, w))
    best = [INF] * n
    best[src] = 0
    queue = deque([(src, 0)])
    for level in range(1, k + 2):
        if not queue:
            break
        print(f"--- tầng {level}: queue = {list(queue)}")
        for _ in range(len(queue)):
            u, cost = queue.popleft()
            for v, w in adj[u]:
                if cost + w < best[v]:
                    best[v] = cost + w
                    queue.append((v, cost + w))
                    print(f"   {u} -> {v}: {cost} + {w} = {cost + w} cải thiện best[{v}]")
        print(f"   best = {best}")
    return -1 if best[dst] == INF else best[dst]

print("đáp án:", bfs_levels_demo(*examples[0][:5]))

--- tầng 1: queue = [(0, 0)]
   0 -> 1: 0 + 100 = 100 cải thiện best[1]
   best = [0, 100, inf, inf]
--- tầng 2: queue = [(1, 100)]
   1 -> 2: 100 + 100 = 200 cải thiện best[2]
   1 -> 3: 100 + 600 = 700 cải thiện best[3]
   best = [0, 100, 200, 700]
đáp án: 700


**Code nộp LeetCode (cách 3):**

In [14]:
class Solution3:
    def findCheapestPrice(self, n: int, flights: List[List[int]], src: int, dst: int, k: int) -> int:
        adj = defaultdict(list)
        for u, v, w in flights:
            adj[u].append((v, w))
        best = [float("inf")] * n
        best[src] = 0
        queue = deque([(src, 0)])
        for _ in range(k + 1):
            if not queue:
                break
            for _ in range(len(queue)):
                u, cost = queue.popleft()
                for v, w in adj[u]:
                    if cost + w < best[v]:
                        best[v] = cost + w
                        queue.append((v, cost + w))
        return -1 if best[dst] == float("inf") else best[dst]


for n, flights, src, dst, k, expected in examples:
    assert Solution3().findCheapestPrice(n, flights, src, dst, k) == expected
print("Cách 3 OK")

Cách 3 OK


**Độ phức tạp:** Time `O((k + 1) * E)` trường hợp xấu (mỗi tầng có thể xét hết cạnh), thường nhanh hơn Bellman-Ford vì chỉ mở rộng các đỉnh vừa được cải thiện. Space `O(n + E)` (danh sách kề + hàng đợi).

Vì sao đúng khi so với `best` chung? Một đường tới `v` bằng `t` chuyến chỉ bị bỏ nếu đã có đường tới `v` **rẻ hơn hoặc bằng** với **không nhiều hơn `t`** chuyến (vì `best[v]` chỉ cập nhật ở tầng không lớn hơn); đường đó thay thế được (nó dùng ít chuyến hơn nên còn nhiều "ngân sách chuyến" để đi tiếp, và rẻ hơn). Do đó bỏ không mất đáp án.

### 7.3. Cách 4 — Dijkstra với trạng thái `(giá, thành phố, số chuyến đã dùng)`

**Ý tưởng khác:** dùng **hàng đợi ưu tiên** theo giá. Mỗi phần tử là `(cost, u, e)` với `e` là số chuyến đã bay. Lấy ra phần tử giá nhỏ nhất; **lần đầu** lấy ra `dst` chính là đáp án (giá nhỏ nhất trong các trạng thái có `e <= k + 1`).

**Giải thích từng đoạn:**

1. Đẩy `(0, src, 0)`. Mảng `fewest[u]` = số chuyến ít nhất mà ta đã **lấy ra** tại `u` (ban đầu `inf`).
2. Lấy `(cost, u, e)`. Nếu `u == dst` → trả `cost`.
3. Nếu `e == k + 1` (hết ngân sách chuyến) hoặc `e >= fewest[u]` (đã từng lấy ra `u` với giá **không lớn hơn** và số chuyến **không nhiều hơn** ⇒ trạng thái này bị lấn át) thì bỏ qua. Ngược lại đặt `fewest[u] = e`.
4. Với mỗi chuyến `u → v, w` đẩy `(cost + w, v, e + 1)`.

**Cắt tỉa đúng không?** Vì ta lấy theo giá tăng dần, phần tử lấy ra sau tại cùng `u` có giá **không nhỏ hơn**. Nếu nó cũng dùng **không ít chuyến hơn** thì nó không thể tốt hơn trạng thái đã lấy trước — bỏ được.

**Code demo:**

In [15]:
def dijkstra_demo(n, flights, src, dst, k):
    adj = defaultdict(list)
    for u, v, w in flights:
        adj[u].append((v, w))
    heap = [(0, src, 0)]
    fewest = [INF] * n
    while heap:
        cost, u, e = heapq.heappop(heap)
        print(f"lấy (giá {cost}, thành phố {u}, {e} chuyến)")
        if u == dst:
            print("   tới dst -> đáp án", cost)
            return cost
        if e == k + 1 or e >= fewest[u]:
            print("   bỏ (hết chuyến hoặc bị lấn át)")
            continue
        fewest[u] = e
        for v, w in adj[u]:
            heapq.heappush(heap, (cost + w, v, e + 1))
    return -1

dijkstra_demo(*examples[0][:5])

lấy (giá 0, thành phố 0, 0 chuyến)
lấy (giá 100, thành phố 1, 1 chuyến)
lấy (giá 200, thành phố 2, 2 chuyến)
   bỏ (hết chuyến hoặc bị lấn át)
lấy (giá 700, thành phố 3, 2 chuyến)
   tới dst -> đáp án 700


700

**Code nộp LeetCode (cách 4):**

In [16]:
class Solution4:
    def findCheapestPrice(self, n: int, flights: List[List[int]], src: int, dst: int, k: int) -> int:
        adj = defaultdict(list)
        for u, v, w in flights:
            adj[u].append((v, w))
        heap = [(0, src, 0)]
        fewest = [float("inf")] * n
        while heap:
            cost, u, e = heapq.heappop(heap)
            if u == dst:
                return cost
            if e == k + 1 or e >= fewest[u]:
                continue
            fewest[u] = e
            for v, w in adj[u]:
                heapq.heappush(heap, (cost + w, v, e + 1))
        return -1


for n, flights, src, dst, k, expected in examples:
    assert Solution4().findCheapestPrice(n, flights, src, dst, k) == expected
print("Cách 4 OK")

Cách 4 OK


**Độ phức tạp:** mỗi đỉnh `u` được "mở rộng" tối đa `k + 1` lần (mỗi lần với `e` nhỏ hơn lần trước) nên số trạng thái `O(n * (k + 1))`, mỗi trạng thái duyệt các cạnh ra: tổng `O((k + 1) * E * log)` thời gian; Space `O(n * (k + 1))` cho hàng đợi. Cách này **dừng sớm** khi gặp `dst`, nên với đồ thị lớn mà đáp án rẻ thường nhanh hơn cách 2.

### 7.4. Cách 5 — Đệ quy có nhớ (top-down DP), tính ngược từ `dst`

**Ý tưởng khác:** định nghĩa `f(v, t)` = giá rẻ nhất để đến `v` từ `src` dùng **tối đa `t` chuyến**. Tính ngược: nếu `v == src` thì `f = 0`; nếu `t == 0` mà `v != src` thì `inf`; còn lại `f(v, t) = min( f(u, t - 1) + w )` trên mọi chuyến **vào** `v` (`u → v, w`).

**Giải thích từng đoạn:**

1. Lập danh sách **chuyến vào** `incoming[v] = [(u, w), ...]`.
2. `f(v, t)` có nhớ bằng `lru_cache` — số trạng thái `n * (k + 2)`.
3. Độ sâu đệ quy tối đa `k + 1 <= 100`, an toàn với giới hạn của Python.

**Code demo:**

In [17]:
def memo_demo(n, flights, src, dst, k):
    incoming = defaultdict(list)
    for u, v, w in flights:
        incoming[v].append((u, w))

    @lru_cache(maxsize=None)
    def f(v, t):
        if v == src:
            return 0
        if t == 0:
            return INF
        best = INF
        for u, w in incoming[v]:
            best = min(best, f(u, t - 1) + w)
        print(f"   f({v}, {t}) = {best}")
        return best

    ans = f(dst, k + 1)
    return -1 if ans == INF else ans

print("đáp án:", memo_demo(*examples[0][:5]))

   f(1, 1) = 100
   f(2, 1) = inf
   f(3, 2) = 700
đáp án: 700


**Code nộp LeetCode (cách 5):**

In [18]:
class Solution5:
    def findCheapestPrice(self, n: int, flights: List[List[int]], src: int, dst: int, k: int) -> int:
        incoming = defaultdict(list)
        for u, v, w in flights:
            incoming[v].append((u, w))

        @lru_cache(maxsize=None)
        def f(v, t):
            if v == src:
                return 0
            if t == 0:
                return float("inf")
            best = float("inf")
            for u, w in incoming[v]:
                best = min(best, f(u, t - 1) + w)
            return best

        ans = f(dst, k + 1)
        return -1 if ans == float("inf") else ans


for n, flights, src, dst, k, expected in examples:
    assert Solution5().findCheapestPrice(n, flights, src, dst, k) == expected
print("Cách 5 OK")

Cách 5 OK


**Độ phức tạp:** số trạng thái `n * (k + 2)`, mỗi trạng thái duyệt các chuyến vào ⇒ tổng `O((k + 1) * E)` thời gian; Space `O(n * k)` cho bộ nhớ đệm và ngăn xếp `O(k)`. Đây chính là bảng `dp[t][v]` ở mục 6 viết theo kiểu top-down (chỉ tính các trạng thái cần).

## 8. TRACE từng bước

### 8.1. Trace cách 2 (Bellman-Ford) — ví dụ 1 với `k = 1` và với `k = 2`

Ta in từng vòng, từng chuyến được xét và nhìn `prev` (đầu vòng) so với `dist` (đang cập nhật). Bảng trạng thái:

| Cột | Ý nghĩa |
|---|---|
| `vòng` | số chuyến tối đa đang cho phép |
| `chuyến (u→v, w)` | cạnh đang xét |
| `prev[u] + w` | giá nếu bay thêm chuyến này |
| `dist` | mảng giá sau khi xét xong chuyến |

In [19]:
def trace_bellman_ford(n, flights, src, dst, k):
    dist = [INF] * n
    dist[src] = 0
    print(f"k = {k} -> {k + 1} vòng. Khởi tạo dist = {dist}")
    for t in range(1, k + 2):
        prev = dist[:]
        print(f"=== vòng {t}: prev = {prev}")
        for u, v, w in flights:
            cand = prev[u] + w
            if cand < dist[v]:
                dist[v] = cand
                print(f"  {u}->{v} ({w}): prev[{u}] + {w} = {cand} < dist[{v}] -> cập nhật, dist = {dist}")
            else:
                print(f"  {u}->{v} ({w}): prev[{u}] + {w} = {cand} không cải thiện")
    return -1 if dist[dst] == INF else dist[dst]

n, flights, src, dst, k, _ = examples[0]
print("kết quả k = 1:", trace_bellman_ford(n, flights, src, dst, 1))

k = 1 -> 2 vòng. Khởi tạo dist = [0, inf, inf, inf]
=== vòng 1: prev = [0, inf, inf, inf]
  0->1 (100): prev[0] + 100 = 100 < dist[1] -> cập nhật, dist = [0, 100, inf, inf]
  1->2 (100): prev[1] + 100 = inf không cải thiện
  2->0 (100): prev[2] + 100 = inf không cải thiện
  1->3 (600): prev[1] + 600 = inf không cải thiện
  2->3 (200): prev[2] + 200 = inf không cải thiện
=== vòng 2: prev = [0, 100, inf, inf]
  0->1 (100): prev[0] + 100 = 100 không cải thiện
  1->2 (100): prev[1] + 100 = 200 < dist[2] -> cập nhật, dist = [0, 100, 200, inf]
  2->0 (100): prev[2] + 100 = inf không cải thiện
  1->3 (600): prev[1] + 600 = 700 < dist[3] -> cập nhật, dist = [0, 100, 200, 700]
  2->3 (200): prev[2] + 200 = inf không cải thiện
kết quả k = 1: 700


Ở vòng 2 chuyến `2→3 (200)` dùng `prev[2]` — đầu vòng 2 `prev[2]` vẫn là `inf` vì `2` chỉ tới được ở vòng 2; nên `200 + inf` không cập nhật. Nhờ `prev`, đường `0→1→2→3` (3 chuyến) **không** lọt vào đáp án khi `k = 1`. So với `k = 2`:

In [20]:
print("kết quả k = 2:", trace_bellman_ford(n, flights, src, dst, 2))

k = 2 -> 3 vòng. Khởi tạo dist = [0, inf, inf, inf]
=== vòng 1: prev = [0, inf, inf, inf]
  0->1 (100): prev[0] + 100 = 100 < dist[1] -> cập nhật, dist = [0, 100, inf, inf]
  1->2 (100): prev[1] + 100 = inf không cải thiện
  2->0 (100): prev[2] + 100 = inf không cải thiện
  1->3 (600): prev[1] + 600 = inf không cải thiện
  2->3 (200): prev[2] + 200 = inf không cải thiện
=== vòng 2: prev = [0, 100, inf, inf]
  0->1 (100): prev[0] + 100 = 100 không cải thiện
  1->2 (100): prev[1] + 100 = 200 < dist[2] -> cập nhật, dist = [0, 100, 200, inf]
  2->0 (100): prev[2] + 100 = inf không cải thiện
  1->3 (600): prev[1] + 600 = 700 < dist[3] -> cập nhật, dist = [0, 100, 200, 700]
  2->3 (200): prev[2] + 200 = inf không cải thiện
=== vòng 3: prev = [0, 100, 200, 700]
  0->1 (100): prev[0] + 100 = 100 không cải thiện
  1->2 (100): prev[1] + 100 = 200 không cải thiện
  2->0 (100): prev[2] + 100 = 300 không cải thiện
  1->3 (600): prev[1] + 600 = 700 không cải thiện
  2->3 (200): prev[2] + 200 = 400 <

Với `k = 2` vòng 3 dùng `prev[2] = 200` (đã có sau vòng 2) nên `dist[3]` cải thiện từ `700` xuống `400`.

### 8.2. Trace cách 4 (Dijkstra có số chuyến) trên ví dụ 1, `k = 1`

Quan sát vì sao `dst` lần đầu lấy ra là `700` chứ không phải `400`:

In [21]:
print("kết quả:", dijkstra_demo(*examples[0][:5]))

lấy (giá 0, thành phố 0, 0 chuyến)
lấy (giá 100, thành phố 1, 1 chuyến)
lấy (giá 200, thành phố 2, 2 chuyến)
   bỏ (hết chuyến hoặc bị lấn át)
lấy (giá 700, thành phố 3, 2 chuyến)
   tới dst -> đáp án 700
kết quả: 700


Trạng thái `(200, 2, 2)` (tới `2` bằng 2 chuyến) không được mở rộng vì `e = 2 = k + 1` (hết chuyến); trạng thái `(400, 3, 3)` (đường `0→1→2→3`) **không bao giờ được tạo ra**; đường `0→1→3` đến `3` với giá `700` và `e = 2` được lấy ra đầu tiên ở `dst`.

### 8.3. Trace cách 5 (đệ quy có nhớ) — dòng `f(v, t)` được tính một lần

In [22]:
print("kết quả:", memo_demo(*examples[1][:5]))

   f(1, 1) = 100
   f(2, 2) = 200
kết quả: 200


Với ví dụ 2 (`0→1→2` giá `200`, `0→2` giá `500`, `k = 1`): `f(2, 2)` xét hai chuyến vào `2`: từ `1` (`f(1, 1) + 100`) và từ `0` (`f(0, 1) + 500`); `f(1, 1)` lại cần `f(0, 0) = 0`. Mỗi `f(v, t)` chỉ in **một lần** vì đã được nhớ.

### 8.4. Trace cách 1 (vét cạn) — đếm số hành trình hợp lệ

In [23]:
def trace_brute(n, flights, src, dst, k):
    adj = defaultdict(list)
    for u, v, w in flights:
        adj[u].append((v, w))
    found = []

    def dfs(u, left, cost, path):
        if u == dst:
            found.append((cost, path))
            return
        if left == 0:
            return
        for v, w in adj[u]:
            dfs(v, left - 1, cost + w, path + [v])

    dfs(src, k + 1, 0, [src])
    for cost, path in sorted(found):
        print(f"  hành trình {path} giá {cost} ({len(path) - 2} điểm dừng)")
    return min(found)[0] if found else -1

print("k = 1:", trace_brute(n, flights, src, dst, 1))
print("k = 2:", trace_brute(n, flights, src, dst, 2))

  hành trình [0, 1, 3] giá 700 (1 điểm dừng)
k = 1: 700
  hành trình [0, 1, 2, 3] giá 400 (2 điểm dừng)
  hành trình [0, 1, 3] giá 700 (1 điểm dừng)
k = 2: 400


## 9. Code nộp LeetCode

Chọn **cách 2 (Bellman-Ford `k + 1` vòng)**: ngắn, `O((k + 1) * E)`, đúng với mọi ràng buộc (`n <= 100`). Đây là code đúng class/method/signature:

In [24]:
from typing import List


class Solution:
    def findCheapestPrice(self, n: int, flights: List[List[int]], src: int, dst: int, k: int) -> int:
        INF_ = float("inf")
        dist = [INF_] * n
        dist[src] = 0
        for _ in range(k + 1):            # tối đa k điểm dừng = k + 1 chuyến bay
            prev = dist[:]                # bản chụp đầu vòng: mỗi vòng chỉ thêm một chuyến
            for u, v, w in flights:
                if prev[u] + w < dist[v]:
                    dist[v] = prev[u] + w
        return -1 if dist[dst] == INF_ else dist[dst]


for n, flights, src, dst, k, expected in examples:
    got = Solution().findCheapestPrice(n, flights, src, dst, k)
    print(got, "(kỳ vọng", expected, ")")
    assert got == expected

700 (kỳ vọng 700 )
200 (kỳ vọng 200 )
500 (kỳ vọng 500 )


## 10. Vì sao thuật toán đúng

**Bellman-Ford theo vòng (cách 2).** Định nghĩa `D_t[v]` = giá nhỏ nhất của một hành trình từ `src` đến `v` dùng **tối đa `t` chuyến** (`inf` nếu không có). Khẳng định: sau `t` vòng, `dist = D_t`. Quy nạp theo `t`:

- `t = 0`: `D_0[src] = 0`, các đỉnh khác `inf`. ✓
- Bước `t - 1 → t`: một hành trình tối ưu đến `v` với tối đa `t` chuyến hoặc dùng tối đa `t - 1` chuyến (giá `D_{t-1}[v]`, mà `dist[v]` ban đầu của vòng đã bằng giá này), hoặc dùng đúng `t` chuyến với chuyến cuối `u → v` giá `w`; khi đó phần trước là hành trình đến `u` với tối đa `t - 1` chuyến, giá tối thiểu `D_{t-1}[u]`, nên giá tối ưu là `D_{t-1}[u] + w`. Vòng lặp lấy `min` của `dist[v]` (đã bằng `D_{t-1}[v]`) với mọi `prev[u] + w` (`prev = D_{t-1}`), đúng bằng `D_t[v]`. ✓

Vai trò của `prev`: nếu đọc `dist[u]` thay vì `prev[u]`, một vòng có thể nối nhiều chuyến liên tiếp (vì `dist[u]` vừa được cập nhật trong cùng vòng), tức dùng quá số chuyến cho phép.

Đáp án là `D_{k+1}[dst]` (`k` điểm dừng = `k + 1` chuyến).

**Hành trình lặp thành phố có làm sai không?** `D_t` định nghĩa trên mọi hành trình (kể cả lặp); vì giá chuyến dương, hành trình tối ưu không lặp, và định nghĩa này cho cùng đáp án như "đường đi đơn" tối ưu.

**BFS theo tầng (cách 3).** Mỗi tầng ứng với một vòng Bellman-Ford chỉ xét các đỉnh vừa được cải thiện (những đỉnh khác không thể cải thiện đỉnh kề). Việc loại một đường không cải thiện `best[v]` là an toàn (đường đã có dùng ≤ số chuyến và giá ≤).

**Dijkstra (cách 4).** Trạng thái `(u, e)` với giá tích lũy; lấy theo giá tăng dần nên lần đầu lấy `dst` có giá nhỏ nhất trong các hành trình `≤ k + 1` chuyến (mọi hành trình rẻ hơn đã được lấy ra trước, và hành trình hợp lệ nào cũng sinh ra trạng thái tương ứng trừ khi bị cắt tỉa bởi trạng thái lấn át). Cắt tỉa an toàn như đã chứng minh ở mục 7.3.

**Đệ quy có nhớ (cách 5).** `f(v, t)` thỏa chính công thức `D_t[v]` (tính ngược từ `v`), nên bằng `D_t[v]`; `f(src, t) = 0` vì không cần bay.

**Vét cạn.** Duyệt mọi hành trình ≤ `k + 1` chuyến; đúng theo định nghĩa.

## 11. Time / Space Complexity

(`E` là số chuyến bay, `n` số thành phố.)

| Cách | Time | Space |
|---|---|---|
| 1. DFS vét cạn | `O((n - 1)^(k + 1))` | `O(k)` |
| 2. Bellman-Ford `k + 1` vòng (nộp) | `O((k + 1) * E)` | `O(n)` |
| 3. BFS theo tầng | `O((k + 1) * E)` | `O(n + E)` |
| 4. Dijkstra có số chuyến | `O((k + 1) * E * log)` | `O(n * (k + 1))` |
| 5. Đệ quy có nhớ | `O((k + 1) * E)` | `O(n * k)` |

Với `n = 100`, đồ thị đầy đủ (`E = 4950`) và `k = 99`: cách 2 cỡ `5 * 10^5` phép. Đo thực tế:

In [25]:
def random_graph(rng, n, m):
    """m chuyến bay, KHÔNG có hai chuyến giữa cùng hai thành phố (kể cả ngược chiều)."""
    pairs = [(a, b) for a in range(n) for b in range(a + 1, n)]
    chosen = rng.sample(pairs, m)
    return [[a, b, rng.randint(1, 10**4)] if rng.random() < 0.5 else [b, a, rng.randint(1, 10**4)] for a, b in chosen]


rng = random.Random(787)
n = 100
flights_big = random_graph(rng, n, n * (n - 1) // 2)      # đồ thị đầy đủ (mỗi cặp một chuyến)
print("số chuyến:", len(flights_big))
for k in (3, 20, 99):
    row, answers = [], set()
    for label, cls in (("cách 2", Solution2), ("cách 3", Solution3), ("cách 4", Solution4), ("cách 5", Solution5)):
        t0 = time.perf_counter()
        ans = cls().findCheapestPrice(n, flights_big, 0, n - 1, k)
        ms = (time.perf_counter() - t0) * 1000
        answers.add(ans)
        row.append(f"{label} {ms:7.1f} ms")
    assert len(answers) == 1
    print(f"k = {k:>2}: đáp án {answers.pop():>5} | " + " | ".join(row))

số chuyến: 4950
k =  3: đáp án  1349 | cách 2     1.0 ms | cách 3     1.4 ms | cách 4     1.1 ms | cách 5     3.4 ms
k = 20: đáp án  1219 | cách 2     5.4 ms | cách 3     2.5 ms | cách 4     1.2 ms | cách 5    13.4 ms


k = 99: đáp án  1219 | cách 2    26.7 ms | cách 3     2.5 ms | cách 4     1.5 ms | cách 5    84.0 ms

Các cách đều chạy tốt với `n = 100`.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Ta viết `check_all` chạy mọi cách (vét cạn chỉ cho đồ thị nhỏ) và đòi đồng ý với một **oracle độc lập** viết theo hướng khác: nhân ma trận kiểu (min, +) — `M_t[u][v]` = giá rẻ nhất từ `u` đến `v` dùng **đúng** `t` chuyến (cho phép lặp thành phố); đáp án là `min` theo `t = 1..k+1` của `M_t[src][dst]`.

In [26]:
def oracle(n, flights, src, dst, k):
    """Oracle: tích ma trận (min, +). Không dùng danh sách kề, không dùng dist theo vòng."""
    W = [[INF] * n for _ in range(n)]
    for u, v, w in flights:
        W[u][v] = w
    cur = W                                   # đúng 1 chuyến
    best = cur[src][dst]
    for _ in range(k):
        nxt = [[INF] * n for _ in range(n)]
        for a in range(n):
            row = cur[a]
            for b in range(n):
                wb = W[b]
                for c in range(n):
                    if wb[c] != INF and row[b] + wb[c] < nxt[a][c]:
                        nxt[a][c] = row[b] + wb[c]
        cur = nxt                             # đúng t chuyến
        best = min(best, cur[src][dst])
    return -1 if best == INF else best


solvers = [
    ("cách 2", Solution2().findCheapestPrice),
    ("cách 3", Solution3().findCheapestPrice),
    ("cách 4", Solution4().findCheapestPrice),
    ("cách 5", Solution5().findCheapestPrice),
]


def check_all(n, flights, src, dst, k, expected=None, brute=True, use_oracle=True):
    results = {name: f(n, [fl[:] for fl in flights], src, dst, k) for name, f in solvers}
    if brute:
        results["vét cạn"] = cheapest_brute(n, flights, src, dst, k)
    if use_oracle:
        results["oracle"] = oracle(n, flights, src, dst, k)
    values = set(results.values())
    assert len(values) == 1, (n, flights, src, dst, k, results)
    got = values.pop()
    if expected is not None:
        assert got == expected, (got, expected)
    return got


for n, flights, src, dst, k, expected in examples:
    check_all(n, flights, src, dst, k, expected)
print("ví dụ đề: OK")

edge = [
    (2, [], 0, 1, 0, -1),                                   # không có chuyến bay nào
    (2, [[0, 1, 5]], 0, 1, 0, 5),                           # bay thẳng, k = 0
    (2, [[1, 0, 5]], 0, 1, 1, -1),                          # chuyến bay sai chiều
    (3, [[0, 1, 1], [1, 2, 1]], 0, 2, 0, -1),               # cần 1 điểm dừng nhưng k = 0
    (3, [[0, 1, 1], [1, 2, 1]], 0, 2, 1, 2),                # vừa đủ
    (3, [[0, 1, 1], [1, 2, 1]], 0, 2, 2, 2),                # k thừa
    (3, [[0, 1, 1], [1, 2, 1], [0, 2, 5]], 0, 2, 0, 5),     # k = 0 buộc bay thẳng dù đắt
    (3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 1, 200),
    (4, [[0, 1, 1], [1, 2, 1], [2, 3, 1], [0, 3, 10]], 0, 3, 1, 10),   # 2 điểm dừng mới rẻ, k = 1 không đủ
    (4, [[0, 1, 1], [1, 2, 1], [2, 3, 1], [0, 3, 10]], 0, 3, 2, 3),
    (4, [[0, 1, 1], [1, 2, 1], [2, 3, 1], [0, 3, 10]], 0, 3, 3, 3),
    (3, [[0, 1, 4], [1, 2, 4]], 2, 0, 2, -1),               # đi ngược chiều: không tới được
    (5, [[0, 1, 1], [0, 2, 5], [1, 3, 1], [2, 3, 1], [3, 4, 1]], 0, 4, 2, 3),
]
for n, flights, src, dst, k, expected in edge:
    check_all(n, flights, src, dst, k, expected)
print("edge case: OK")

ví dụ đề: OK
edge case: OK


### 12.2. Phản ví dụ cho các cách sai

Ta dựng các cách giải sai nghe có lý rồi tìm input chứng minh chúng sai:

In [27]:
def wrong_dijkstra_plain(n, flights, src, dst, k):
    """SAI: Dijkstra thông thường, bỏ qua giới hạn số điểm dừng."""
    adj = defaultdict(list)
    for u, v, w in flights:
        adj[u].append((v, w))
    dist = [INF] * n
    dist[src] = 0
    heap = [(0, src)]
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]:
            continue
        for v, w in adj[u]:
            if d + w < dist[v]:
                dist[v] = d + w
                heapq.heappush(heap, (d + w, v))
    return -1 if dist[dst] == INF else dist[dst]


def wrong_visited_dijkstra(n, flights, src, dst, k):
    """SAI: Dijkstra với 'đã thăm' theo thành phố (mất đường ít chuyến hơn nhưng đắt hơn)."""
    adj = defaultdict(list)
    for u, v, w in flights:
        adj[u].append((v, w))
    heap = [(0, src, 0)]
    visited = set()
    while heap:
        cost, u, e = heapq.heappop(heap)
        if u == dst:
            return cost
        if u in visited:
            continue
        visited.add(u)                 # đánh dấu theo thành phố, bất kể đã dùng bao nhiêu chuyến
        if e == k + 1:
            continue
        for v, w in adj[u]:
            heapq.heappush(heap, (cost + w, v, e + 1))
    return -1


def wrong_one_less(n, flights, src, dst, k):
    """SAI: chạy k vòng thay vì k + 1."""
    dist = [INF] * n
    dist[src] = 0
    for _ in range(k):
        prev = dist[:]
        for u, v, w in flights:
            dist[v] = min(dist[v], prev[u] + w)
    return -1 if dist[dst] == INF else dist[dst]


tests = [
    ("ví dụ 1", examples[0]),
    ("k = 0 bay thẳng đắt", (3, [[0, 1, 1], [1, 2, 1], [0, 2, 5]], 0, 2, 0, 5)),
    ("cần đúng k + 1 chuyến", (3, [[0, 1, 1], [1, 2, 1]], 0, 2, 1, 2)),
]
for label, (n, flights, src, dst, k, expected) in tests:
    print(f"{label:22}: đúng = {expected:>4} | Dijkstra thường = {wrong_dijkstra_plain(n, flights, src, dst, k):>4}"
          f" | 'đã thăm' = {wrong_visited_dijkstra(n, flights, src, dst, k):>4} | k vòng = {wrong_one_less(n, flights, src, dst, k):>4}")

assert wrong_dijkstra_plain(*examples[0][:5]) != 700
assert wrong_one_less(3, [[0, 1, 1], [1, 2, 1]], 0, 2, 1) != 2
# phản ví dụ cho "đã thăm": đường 0->1->2 rẻ nhưng nhiều chuyến chặn đường 0->2 đắt mà ít chuyến
fl = [[0, 1, 1], [1, 2, 1], [0, 2, 50], [2, 3, 1]]
print("phản ví dụ 'đã thăm':", wrong_visited_dijkstra(4, fl, 0, 3, 1), "vs đúng", Solution().findCheapestPrice(4, fl, 0, 3, 1))
assert wrong_visited_dijkstra(4, fl, 0, 3, 1) != Solution().findCheapestPrice(4, fl, 0, 3, 1)

ví dụ 1               : đúng =  700 | Dijkstra thường =  400 | 'đã thăm' =  700 | k vòng =   -1
k = 0 bay thẳng đắt   : đúng =    5 | Dijkstra thường =    2 | 'đã thăm' =    5 | k vòng =   -1
cần đúng k + 1 chuyến : đúng =    2 | Dijkstra thường =    2 | 'đã thăm' =    2 | k vòng =   -1
phản ví dụ 'đã thăm': -1 vs đúng 51


- **Dijkstra thường** trả `400` cho ví dụ 1 (bỏ qua `k = 1`).
- **Chạy `k` vòng** (thay vì `k + 1`) cho `-1` ở `[[0,1,1],[1,2,1]]` với `k = 1`: cần 2 chuyến.
- **"Đã thăm" theo thành phố**: thành phố `2` bị đánh dấu khi tới bằng đường rẻ nhưng nhiều chuyến (`0→1→2`), làm mất đường `0→2` (đắt nhưng chỉ 1 chuyến) cần thiết khi ngân sách chuyến hạn chế (ví dụ phản chứng `[[0,1,1],[1,2,1],[0,2,50],[2,3,1]]`, `k = 1`: đúng `51`, sai `-1`).

### 12.3. Đối chiếu nhiều input ngẫu nhiên (seed cố định)

Sinh đồ thị ngẫu nhiên **đúng ràng buộc** (`n` nhỏ, số chuyến tối đa `n(n-1)/2`, giá `1..10^4`, không có hai chuyến giữa cùng hai thành phố, `src != dst`, `0 <= k < n`) và so mọi cách (kể cả vét cạn và oracle ma trận).

In [28]:
rng = random.Random(20787)
reachable = 0
total = 0
for _ in range(4000):
    n = rng.randint(2, 7)
    m = rng.randint(0, n * (n - 1) // 2)
    flights = random_graph(rng, n, m)
    src, dst = rng.sample(range(n), 2)
    k = rng.randint(0, n - 1)
    got = check_all(n, flights, src, dst, k, brute=(n <= 5 or k <= 2))
    total += 1
    reachable += got != -1
print(f"{total} đồ thị nhỏ: mọi cách + vét cạn + oracle khớp ({reachable} có hành trình, {total - reachable} trả -1)")

# giá nhỏ để có nhiều "hòa" và nhiều đường khác số chuyến
for _ in range(1500):
    n = rng.randint(2, 7)
    m = rng.randint(1, n * (n - 1) // 2)
    pairs = [(a, b) for a in range(n) for b in range(a + 1, n)]
    flights = [[a, b, rng.randint(1, 4)] if rng.random() < 0.5 else [b, a, rng.randint(1, 4)] for a, b in rng.sample(pairs, m)]
    src, dst = rng.sample(range(n), 2)
    k = rng.randint(0, n - 1)
    check_all(n, flights, src, dst, k, brute=(n <= 5 or k <= 2))
print("1500 đồ thị giá nhỏ: OK")

4000 đồ thị nhỏ: mọi cách + vét cạn + oracle khớp (1454 có hành trình, 2546 trả -1)
1500 đồ thị giá nhỏ: OK


In [29]:
# đồ thị lớn: n = 30..100, đối chiếu 4 cách nhanh với oracle (oracle chạy O(k n^3) nên chỉ dùng n <= 30)
rng = random.Random(7)
for _ in range(60):
    n = rng.randint(8, 100)
    m = rng.randint(n, n * (n - 1) // 2)
    flights = random_graph(rng, n, m)
    src, dst = rng.sample(range(n), 2)
    k = rng.randint(0, n - 1)
    vals = {f(n, [fl[:] for fl in flights], src, dst, k) for _, f in solvers}
    assert len(vals) == 1, (n, k, vals)
print("60 đồ thị lớn (n tới 100): các cách nhanh đồng ý")

for _ in range(12):
    n = rng.randint(8, 24)
    flights = random_graph(rng, n, rng.randint(n, n * (n - 1) // 2))
    src, dst = rng.sample(range(n), 2)
    k = rng.randint(0, n - 1)
    check_all(n, flights, src, dst, k, brute=False)
print("12 đồ thị vừa: khớp cả oracle ma trận")

# đường dài nhất: chuỗi 100 thành phố, phải dùng đúng k = n - 2 điểm dừng
chain = [[i, i + 1, 1] for i in range(99)]
assert Solution().findCheapestPrice(100, chain, 0, 99, 98) == 99
assert Solution().findCheapestPrice(100, chain, 0, 99, 97) == -1
print("chuỗi 100 thành phố: OK")

60 đồ thị lớn (n tới 100): các cách nhanh đồng ý
12 đồ thị vừa: khớp cả oracle ma trận
chuỗi 100 thành phố: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng Dijkstra thông thường (bỏ qua số điểm dừng)

**Code sai:** đã dựng `wrong_dijkstra_plain` ở mục 12.2.

In [30]:
n, flights, src, dst, k, expected = examples[0]
print("sai  :", wrong_dijkstra_plain(n, flights, src, dst, k), "(đường 0->1->2->3 dùng 2 điểm dừng)")
print("đúng :", Solution().findCheapestPrice(n, flights, src, dst, k))

sai  : 400 (đường 0->1->2->3 dùng 2 điểm dừng)
đúng : 700


**Vì sao sai:** Dijkstra chỉ tối ưu **giá**; bài còn có ràng buộc **số cạnh** nên "đường rẻ nhất tới một thành phố" không đủ. Trạng thái phải gồm cả số chuyến đã dùng (cách 4) hoặc phải xử lý theo vòng (cách 2).

### Lỗi 2: Quên bản chụp `prev` trong Bellman-Ford

In [31]:
class Wrong2:
    def findCheapestPrice(self, n, flights, src, dst, k):
        dist = [INF] * n
        dist[src] = 0
        for _ in range(k + 1):
            for u, v, w in flights:                 # SAI: đọc/ghi cùng mảng dist
                if dist[u] + w < dist[v]:
                    dist[v] = dist[u] + w
        return -1 if dist[dst] == INF else dist[dst]

n, flights, src, dst, k, expected = examples[0]
print("sai  :", Wrong2().findCheapestPrice(n, flights, src, dst, k), "(đã đi 3 chuyến trong 2 vòng)")
print("đúng :", Solution().findCheapestPrice(n, flights, src, dst, k))

sai  : 400 (đã đi 3 chuyến trong 2 vòng)
đúng : 700


**Vì sao sai:** không có `prev`, một vòng có thể **nối nhiều chuyến**: `0→1` cập nhật `dist[1]`, rồi cạnh `1→2` ngay sau đó dùng giá mới, ... cho nên "tối đa `k + 1` chuyến" bị phá vỡ (kết quả phụ thuộc cả thứ tự các cạnh trong `flights`). Phải đọc từ bản chụp `prev` đầu vòng.

### Lỗi 3: Nhầm `k` điểm dừng với `k` chuyến bay (lệch 1)

In [32]:
for kk in range(0, 3):
    print(f"k = {kk}: k vòng (sai) -> {wrong_one_less(3, [[0, 1, 1], [1, 2, 1]], 0, 2, kk):>3} | k + 1 vòng (đúng) -> {Solution().findCheapestPrice(3, [[0, 1, 1], [1, 2, 1]], 0, 2, kk):>3}")

k = 0: k vòng (sai) ->  -1 | k + 1 vòng (đúng) ->  -1
k = 1: k vòng (sai) ->  -1 | k + 1 vòng (đúng) ->   2
k = 2: k vòng (sai) ->   2 | k + 1 vòng (đúng) ->   2


**Vì sao sai:** `k` là số **điểm dừng**, không phải số chuyến. Đi từ `0` tới `2` qua `1` có **1 điểm dừng** và **2 chuyến**. Số vòng phải là `k + 1`.

### Lỗi 4: "Đã thăm" theo thành phố làm mất đường ít chuyến hơn

In [33]:
fl = [[0, 1, 1], [1, 2, 1], [0, 2, 50], [2, 3, 1]]
print("sai  :", wrong_visited_dijkstra(4, fl, 0, 3, 1))
print("đúng :", Solution().findCheapestPrice(4, fl, 0, 3, 1))
print("kiểm tra bằng oracle:", oracle(4, fl, 0, 3, 1))

sai  : -1
đúng : 51
kiểm tra bằng oracle: 51


**Vì sao sai:** với `k = 1` (tối đa 2 chuyến), thành phố `2` được đánh dấu khi tới bằng `0→1→2` (giá 2, đã dùng 2 chuyến) — nhưng từ đó không còn chuyến nào để bay tiếp tới `3`; đường `0→2→3` (giá 51, 2 chuyến) bị chặn vì `2` đã "thăm". Đúng ra ta chỉ được loại trạng thái bị **lấn át cả về giá lẫn số chuyến** (cách 4).

### Lỗi 5: Quên đổi `inf` thành `-1`

In [34]:
class Wrong5:
    def findCheapestPrice(self, n, flights, src, dst, k):
        dist = [INF] * n
        dist[src] = 0
        for _ in range(k + 1):
            prev = dist[:]
            for u, v, w in flights:
                if prev[u] + w < dist[v]:
                    dist[v] = prev[u] + w
        return dist[dst]                            # SAI: trả inf khi không tới được

print("sai  :", Wrong5().findCheapestPrice(2, [], 0, 1, 0))
print("đúng :", Solution().findCheapestPrice(2, [], 0, 1, 0))

sai  : inf
đúng : -1


**Vì sao sai:** đề yêu cầu `-1` khi không có hành trình. `float("inf")` không phải `-1` (và không phải `int`). Luôn đổi ở cuối.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "đường đi ngắn nhất **với tối đa K cạnh/bước**" → **Bellman-Ford giới hạn vòng** (`k + 1` vòng), hoặc Dijkstra với **số bước trong trạng thái**. Dijkstra thuần không giải được vì thêm một chiều ràng buộc.
- **Mẫu "DP theo số bước":** `dp[t][v]` = tối ưu khi dùng tối đa `t` bước; chỉ cần mảng của `t - 1` để tính `t` — dùng **bản chụp** `prev`.
- **`k` điểm dừng = `k + 1` chuyến bay**: đọc kỹ định nghĩa đơn vị để khỏi lệch 1.
- **Cắt tỉa an toàn** chỉ khi trạng thái bị lấn át ở **mọi** chiều (giá **và** số bước).
- **Ràng buộc nhỏ** (`n <= 100`, `E <= 4950`) cho phép `O(k * E)` thoải mái.
- **Oracle cho đồ thị:** tích ma trận (min, +) là cách kiểm tra độc lập, dễ viết, đúng cho `n` nhỏ.
- **Python:** `float("inf")` cho "chưa tới"; `dist[:]` để chụp; `heapq` với tuple `(giá, ...)`; `lru_cache` cho top-down.
- **Họ bài liên quan:** Network Delay Time (743), Path With Minimum Effort (1631), Shortest Path in a Grid with Obstacles Elimination (1293), Minimum Cost to Reach Destination in Time (1928).

## 15. Biến thể

1. **Dừng đúng `k` điểm dừng** (không phải "tối đa"): dùng DP theo **đúng** `t` chuyến (không giữ `dp[t - 1]` làm cận).
2. **Truy vết hành trình** (in ra đường đi rẻ nhất) bằng mảng `parent` theo từng vòng.
3. **Bài đối ngẫu:** với ngân sách giá cho trước, tìm **số điểm dừng ít nhất** để đi được.
4. **Network Delay Time (LeetCode 743):** Dijkstra thông thường (không có ràng buộc số bước).
5. **Minimum Cost to Reach Destination in Time (LeetCode 1928):** trạng thái `(thành phố, thời gian)`.
6. **Thêm phí/phạt mỗi điểm dừng** (giá mỗi chuyến cộng một hằng).
7. **Đồ thị có thể có trọng số âm:** Bellman-Ford vẫn đúng với giới hạn vòng (Dijkstra thì không).

Demo biến thể 1 và 2 (đã đối chiếu oracle/vét cạn):

In [35]:
def cheapest_exact_flights(n, flights, src, dst, count):
    """Giá rẻ nhất dùng ĐÚNG count chuyến (cho phép lặp thành phố); -1 nếu không có."""
    dist = [INF] * n
    dist[src] = 0
    for _ in range(count):
        nxt = [INF] * n
        for u, v, w in flights:
            if dist[u] + w < nxt[v]:
                nxt[v] = dist[u] + w
        dist = nxt
    return -1 if dist[dst] == INF else dist[dst]


def cheapest_with_path(n, flights, src, dst, k):
    """Biến thể 2: trả về (giá, hành trình) rẻ nhất với tối đa k điểm dừng."""
    dist = [INF] * n
    path = [None] * n
    dist[src] = 0
    path[src] = [src]
    for _ in range(k + 1):
        prev, prev_path = dist[:], path[:]
        for u, v, w in flights:
            if prev[u] + w < dist[v]:
                dist[v] = prev[u] + w
                path[v] = prev_path[u] + [v]
    return (-1, None) if dist[dst] == INF else (dist[dst], path[dst])


n, flights, src, dst, k, _ = examples[0]
print("đúng 2 chuyến:", cheapest_exact_flights(n, flights, src, dst, 2), "| đúng 3 chuyến:", cheapest_exact_flights(n, flights, src, dst, 3))
print("hành trình k = 1:", cheapest_with_path(n, flights, src, dst, 1))
print("hành trình k = 2:", cheapest_with_path(n, flights, src, dst, 2))

rng = random.Random(15)
for _ in range(500):
    n = rng.randint(2, 6)
    flights = random_graph(rng, n, rng.randint(0, n * (n - 1) // 2))
    src, dst = rng.sample(range(n), 2)
    k = rng.randint(0, n - 1)
    price, route = cheapest_with_path(n, flights, src, dst, k)
    assert price == Solution().findCheapestPrice(n, flights, src, dst, k)
    if route:
        cost = sum(next(w for a, b, w in flights if a == x and b == y) for x, y in zip(route, route[1:]))
        assert cost == price and route[0] == src and route[-1] == dst and len(route) - 2 <= k
print("biến thể truy vết: OK")

đúng 2 chuyến: 700 | đúng 3 chuyến: 400
hành trình k = 1: (700, [0, 1, 3])
hành trình k = 2: (400, [0, 1, 2, 3])
biến thể truy vết: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DFS vét cạn | Thử mọi hành trình ≤ `k + 1` chuyến | `O((n-1)^(k+1))` | `O(k)` | Chỉ kiểm chứng đồ thị nhỏ |
| 2. Bellman-Ford `k + 1` vòng | `dp` theo số chuyến, bản chụp `prev` | `O((k+1) E)` | `O(n)` | **Cách nộp**: ngắn, chắc chắn |
| 3. BFS theo tầng | Chỉ mở rộng đỉnh vừa cải thiện | `O((k+1) E)` | `O(n + E)` | Đồ thị thưa, `k` nhỏ |
| 4. Dijkstra có số chuyến | Hàng đợi ưu tiên, cắt tỉa trạng thái bị lấn át | `O((k+1) E log)` | `O(n (k+1))` | Cần dừng sớm; kiểu "đường ngắn nhất" |
| 5. Đệ quy có nhớ | `f(v, t)` tính ngược từ `dst` | `O((k+1) E)` | `O(n k)` | Dễ suy nghĩ top-down, chỉ tính trạng thái cần |

Về ý tưởng: cách 1 không dùng nhớ; cách 2, 5 là cùng DP `dp[t][v]` (bottom-up vs top-down); cách 3 là cùng DP nhưng chỉ lan từ các đỉnh vừa đổi; cách 4 đổi sang bài đường ngắn nhất trên **đồ thị trạng thái** `(thành phố, số chuyến)`.

In [36]:
# so sánh số trạng thái/thao tác trên đồ thị lớn (đếm số lần xét cạnh) và đối chiếu kết quả
def count_edge_checks_bellman(n, flights, src, dst, k):
    return (k + 1) * len(flights)

def count_pops_dijkstra(n, flights, src, dst, k):
    adj = defaultdict(list)
    for u, v, w in flights:
        adj[u].append((v, w))
    heap, fewest, pops = [(0, src, 0)], [INF] * n, 0
    while heap:
        cost, u, e = heapq.heappop(heap)
        pops += 1
        if u == dst:
            return pops
        if e == k + 1 or e >= fewest[u]:
            continue
        fewest[u] = e
        for v, w in adj[u]:
            heapq.heappush(heap, (cost + w, v, e + 1))
    return pops

rng = random.Random(16)
for n_, m_ in ((30, 60), (60, 300), (100, 1000)):
    fl = random_graph(rng, n_, m_)
    for k in (2, 10):
        a = Solution2().findCheapestPrice(n_, fl, 0, n_ - 1, k)
        b = Solution4().findCheapestPrice(n_, fl, 0, n_ - 1, k)
        assert a == b
        print(f"n = {n_:>3}, E = {m_:>4}, k = {k:>2}: đáp án {a:>6} | Bellman-Ford xét {count_edge_checks_bellman(n_, fl, 0, n_ - 1, k):>6} cạnh | Dijkstra lấy {count_pops_dijkstra(n_, fl, 0, n_ - 1, k):>6} phần tử")

n =  30, E =   60, k =  2: đáp án     -1 | Bellman-Ford xét    180 cạnh | Dijkstra lấy     16 phần tử
n =  30, E =   60, k = 10: đáp án  15033 | Bellman-Ford xét    660 cạnh | Dijkstra lấy     13 phần tử
n =  60, E =  300, k =  2: đáp án   7534 | Bellman-Ford xét    900 cạnh | Dijkstra lấy     21 phần tử
n =  60, E =  300, k = 10: đáp án   7534 | Bellman-Ford xét   3300 cạnh | Dijkstra lấy     27 phần tử
n = 100, E = 1000, k =  2: đáp án  13629 | Bellman-Ford xét   3000 cạnh | Dijkstra lấy    344 phần tử
n = 100, E = 1000, k = 10: đáp án   4539 | Bellman-Ford xét  11000 cạnh | Dijkstra lấy     33 phần tử


## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [37]:
fl = [[0, 1, 5], [1, 2, 5], [0, 2, 20]]
print(Solution().findCheapestPrice(3, fl, 0, 2, 0))
print(Solution().findCheapestPrice(3, fl, 0, 2, 1))
print(Solution().findCheapestPrice(3, fl, 2, 0, 2))

20
10
-1


**Bài 2 (tính tay).** Với ví dụ 1 (`n = 4`, `flights = [[0,1,100],[1,2,100],[2,0,100],[1,3,600],[2,3,200]]`, `src = 0`) hãy tính `dp[t]` (mảng giá tới thành phố `0, 1, 2, 3`) cho `t = 0, 1, 2, 3` theo công thức mục 6 (dùng `inf` cho "chưa tới", ta viết `None`). Gán list các list vào `ans2` (đang là `None`), ví dụ `ans2[0] = [0, None, None, None]`.

In [38]:
ans2 = None   # TODO: list gồm 4 list, mỗi list 4 phần tử (số hoặc None cho inf)

**Bài 3 (điền TODO).** Hoàn thành `bellman_ford_rounds(n, flights, src, rounds)` trả về mảng `dist` sau đúng `rounds` vòng (dùng `inf` cho chưa tới, có bản chụp `prev`).

In [39]:
def bellman_ford_rounds(n, flights, src, rounds):
    # TODO: dist = [INF] * n; dist[src] = 0; lặp rounds lần với prev = dist[:]
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_cheapest(n, flights, src, dst, k)` theo Bellman-Ford `k + 1` vòng.

In [40]:
def my_cheapest(n, flights, src, dst, k):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) Viết `min_stops_under_budget(n, flights, src, dst, budget)`: số **điểm dừng ít nhất** để đi từ `src` đến `dst` với tổng giá `<= budget`; `-1` nếu không thể (gợi ý: chạy từng vòng Bellman-Ford và kiểm tra `dist[dst] <= budget` sau mỗi vòng). (b) Viết `cheapest_with_surcharge(n, flights, src, dst, k, fee)`: giống bài gốc nhưng **mỗi điểm dừng** (thành phố trung gian) tốn thêm phụ phí `fee` (tức mỗi chuyến **sau chuyến đầu** thêm `fee`).

In [41]:
def min_stops_under_budget(n, flights, src, dst, budget):
    # TODO
    pass

def cheapest_with_surcharge(n, flights, src, dst, k, fee):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: `k = 0` chỉ được bay thẳng; `k = 1` được qua 1 thành phố; chiều bay quan trọng.
- Bài 2: dòng `t = 0` chỉ có `dist[0] = 0`; mỗi dòng sau lấy dòng trước rồi thử từng chuyến.
- Bài 3: khớp với "demo" ở mục 7.1.
- Bài 4: `k + 1` vòng, bản chụp `prev`.
- Bài 5: (a) số điểm dừng = số vòng − 1. (b) mỗi chuyến sau chuyến đầu cộng thêm `fee`.

**Mức 2 (vừa):**
- Bài 2: `t = 1`: `[0, 100, None, None]`; `t = 2`: `1 → 2` (giá 200) và `1 → 3` (700).
- Bài 3: trong mỗi vòng `for u, v, w in flights: if prev[u] + w < dist[v]: dist[v] = prev[u] + w`.
- Bài 4: `return -1 if dist[dst] == INF else dist[dst]`.
- Bài 5: (a) `for rounds in range(1, n + 1)`: sau mỗi vòng nếu `dist[dst] <= budget` trả `rounds - 1`. (b) giá một chuyến từ thành phố **khác src**... đơn giản hơn: coi phụ phí thêm vào **mọi chuyến xuất phát từ thành phố khác `src` ở vòng sau**; hãy tự thiết kế trạng thái (nếu `u == src` ở vòng 1 không cộng phí, các vòng sau cộng).

**Mức 3 (gần đáp án):**
- Bài 2: `t = 3`: `dp[3] = [0, 100, 200, 400]` (`400` nhờ đường `0→1→2→3`).
- Bài 3, 4: xem mục 9, code nộp.
- Bài 5: (b) cách gọn: ở vòng `r` (từ 1), giá dùng chuyến `u → v` là `prev[u] + w + (fee if r > 1 else 0)`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [42]:
# Bài 1
fl = [[0, 1, 5], [1, 2, 5], [0, 2, 20]]
assert Solution().findCheapestPrice(3, fl, 0, 2, 0) == 20
assert Solution().findCheapestPrice(3, fl, 0, 2, 1) == 10
assert Solution().findCheapestPrice(3, fl, 2, 0, 2) == -1

# Bài 2
ans2 = [[0, None, None, None],
        [0, 100, None, None],
        [0, 100, 200, 700],
        [0, 100, 200, 400]]
n2, fl2 = examples[0][0], examples[0][1]
dist = [INF] * n2
dist[0] = 0
table = [[None if x == INF else x for x in dist]]
for _ in range(3):
    prev = dist[:]
    for u, v, w in fl2:
        if prev[u] + w < dist[v]:
            dist[v] = prev[u] + w
    table.append([None if x == INF else x for x in dist])
assert ans2 == table, table
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [43]:
# Bài 3
def bellman_ford_rounds(n, flights, src, rounds):
    dist = [INF] * n
    dist[src] = 0
    for _ in range(rounds):
        prev = dist[:]
        for u, v, w in flights:
            if prev[u] + w < dist[v]:
                dist[v] = prev[u] + w
    return dist

assert bellman_ford_rounds(4, examples[0][1], 0, 0) == [0, INF, INF, INF]
assert bellman_ford_rounds(4, examples[0][1], 0, 2) == [0, 100, 200, 700]
assert bellman_ford_rounds(4, examples[0][1], 0, 3) == [0, 100, 200, 400]
print("Bài 3 OK")

Bài 3 OK


In [44]:
# Bài 4
def my_cheapest(n, flights, src, dst, k):
    dist = [INF] * n
    dist[src] = 0
    for _ in range(k + 1):
        prev = dist[:]
        for u, v, w in flights:
            if prev[u] + w < dist[v]:
                dist[v] = prev[u] + w
    return -1 if dist[dst] == INF else dist[dst]

rng = random.Random(4)
for _ in range(500):
    n = rng.randint(2, 7)
    flights = random_graph(rng, n, rng.randint(0, n * (n - 1) // 2))
    src, dst = rng.sample(range(n), 2)
    k = rng.randint(0, n - 1)
    assert my_cheapest(n, flights, src, dst, k) == oracle(n, flights, src, dst, k)
print("Bài 4 OK")

Bài 4 OK


In [45]:
# Bài 5
def min_stops_under_budget(n, flights, src, dst, budget):
    dist = [INF] * n
    dist[src] = 0
    for rounds in range(1, n + 1):          # đường đơn có tối đa n - 1 chuyến
        prev = dist[:]
        for u, v, w in flights:
            if prev[u] + w < dist[v]:
                dist[v] = prev[u] + w
        if dist[dst] <= budget:
            return rounds - 1
    return -1


def cheapest_with_surcharge(n, flights, src, dst, k, fee):
    dist = [INF] * n
    dist[src] = 0
    for r in range(1, k + 2):
        prev = dist[:]
        extra = fee if r > 1 else 0         # chuyến thứ r > 1 xuất phát từ một điểm dừng
        for u, v, w in flights:
            if prev[u] + w + extra < dist[v]:
                dist[v] = prev[u] + w + extra
    return -1 if dist[dst] == INF else dist[dst]


# (a) kiểm tra bằng oracle: số điểm dừng nhỏ nhất k sao cho giá <= budget
rng = random.Random(5)
for _ in range(500):
    n = rng.randint(2, 6)
    flights = random_graph(rng, n, rng.randint(0, n * (n - 1) // 2))
    src, dst = rng.sample(range(n), 2)
    budget = rng.randint(1, 3 * 10**4)
    want = -1
    for k in range(0, n):
        c = oracle(n, flights, src, dst, k)
        if c != -1 and c <= budget:
            want = k
            break
    assert min_stops_under_budget(n, flights, src, dst, budget) == want, (flights, src, dst, budget)

# (b) với fee = 0 trùng bài gốc; so sánh với vét cạn có phụ phí
def brute_surcharge(n, flights, src, dst, k, fee):
    adj = defaultdict(list)
    for u, v, w in flights:
        adj[u].append((v, w))
    best = INF
    def dfs(u, left, cost, used):
        nonlocal best
        if u == dst:
            best = min(best, cost)
            return
        if left == 0:
            return
        for v, w in adj[u]:
            dfs(v, left - 1, cost + w + (fee if used >= 1 else 0), used + 1)
    dfs(src, k + 1, 0, 0)
    return -1 if best == INF else best

for _ in range(500):
    n = rng.randint(2, 6)
    flights = random_graph(rng, n, rng.randint(0, n * (n - 1) // 2))
    src, dst = rng.sample(range(n), 2)
    k = rng.randint(0, n - 1)
    fee = rng.randint(0, 50)
    assert cheapest_with_surcharge(n, flights, src, dst, k, fee) == brute_surcharge(n, flights, src, dst, k, fee)
    assert cheapest_with_surcharge(n, flights, src, dst, k, 0) == Solution().findCheapestPrice(n, flights, src, dst, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template Bellman-Ford giới hạn vòng (nộp):**

```python
class Solution:
    def findCheapestPrice(self, n, flights, src, dst, k):
        INF = float("inf")
        dist = [INF] * n
        dist[src] = 0
        for _ in range(k + 1):             # k điểm dừng = k + 1 chuyến bay
            prev = dist[:]                 # bản chụp đầu vòng
            for u, v, w in flights:
                if prev[u] + w < dist[v]:
                    dist[v] = prev[u] + w
        return -1 if dist[dst] == INF else dist[dst]
```

**Template Dijkstra có số bước:**

```python
heap = [(0, src, 0)]
fewest = [INF] * n
while heap:
    cost, u, e = heappop(heap)
    if u == dst: return cost
    if e == k + 1 or e >= fewest[u]: continue
    fewest[u] = e
    for v, w in adj[u]:
        heappush(heap, (cost + w, v, e + 1))
return -1
```

**Checklist trước khi submit:**

- [ ] Số vòng/bước là `k + 1` (điểm dừng ≠ chuyến bay)?
- [ ] Bellman-Ford có **bản chụp `prev`** để mỗi vòng chỉ thêm một chuyến?
- [ ] Dijkstra: trạng thái có **số chuyến**; cắt tỉa chỉ khi bị lấn át cả giá và số chuyến?
- [ ] Trả `-1` khi `dist[dst]` vẫn là `inf`?
- [ ] Chiều chuyến bay đúng (`from → to`), `src != dst`?
- [ ] Đã test: không có chuyến nào, `k = 0`, `k` thừa, đường rẻ nhưng nhiều chuyến vs đắt nhưng ít chuyến?
- [ ] Đã so với oracle (tích ma trận min-plus) / vét cạn trên đồ thị nhỏ ngẫu nhiên?